In [9]:
#pip install pretty_midi

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from functools import reduce

import random

from pretty_midi import instrument_name_to_program
from pretty_midi import Instrument
from pretty_midi import Note
from pretty_midi import PrettyMIDI

torch.cuda.is_available()

True

In [2]:
# utils
def selfPrecision(x,r):
    if r==0:
        return x
    else:
        return int(float(x)*float(10**r))/float(10**r)

In [3]:
# Midi utils

def createSequence(midiData):
    
    sequence=[]

    for instrument in midiData.instruments:

        for note in instrument.notes:
            
            sequence.append(
                [
                    note.pitch,                  #pitch
                    selfPrecision(note.start,1),    
                    selfPrecision(note.end,1),        
                 ]
                )
        return sequence

    
def OutMidi(sequence,name="000",instrument='Acoustic Grand Piano'):
    
    out = PrettyMIDI()
    program = instrument_name_to_program(instrument)
    Piano = Instrument(program=program)
    
    for j in sequence:
        
        note = Note(
            velocity=100,
            pitch =  j[0],
            start =  j[1],
            end =    j[2]
                    )
        
        Piano.notes.append(note)
    out.instruments.append(Piano)
    out.write(str(name)+'_.mid')
    return out


In [ ]:
#extract data
import glob
mypath ="data/piano/*.mid"
files =glob.glob(mypath) 

sequence=[]
u=0
for midiPath in files:
    if(u%100==0):
        print(int(100*u/len(files)))
    u+=1
    try:
        seq=createSequence(PrettyMIDI(midiPath))
        if seq!=None:  
            if all([x[1]<99.9 for x in seq]) and all([x[2]<99.9 for x in seq]):#seq[-1][2]<99.99:
                sequence.append(seq)
        
    except Exception:
        pass

In [ ]:
# hyperparameters
batch_size = 8 # how many independent sequences will we process in parallel?
block_size = 64 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 100
n_embd = 150
n_head = 12
n_layer = 12
dropout = 0.2
torch.manual_seed(1337)

In [ ]:

#setup note,start,end vocabulary

Note_Chars= sorted(list(set(range(1,128))))
note_vocab_size=128
note_dict_encode={ ch:i for i,ch in enumerate(Note_Chars) }
note_dict_decode= { i:ch for i,ch in enumerate(Note_Chars) }
note_dict_encode[0]=0
note_dict_decode[0]=0

Start_Chars=sorted(list(set([x/10 for x in range(0,1000,1)])))
start_vocab_size=len(Start_Chars)
start_dict_encode={ ch:i for i,ch in enumerate(Start_Chars) }
start_dict_decode= { i:ch for i,ch in enumerate(Start_Chars) }

End_Chars=  sorted(list(set([x/10 for x in range(0,1000,1)])))
end_vocab_size=len(End_Chars)
end_dict_encode={ ch:i for i,ch in enumerate(End_Chars) }
end_dict_decode= { i:ch for i,ch in enumerate(End_Chars) }

def Mencode(x):
    note=[]
    start=[]
    end=[]
    for i in x:
        note.append(note_dict_encode[i[0]])
        start.append(start_dict_encode[i[1]])
        end.append(end_dict_encode[i[2]])
    
    out = torch.stack(
        [
         torch.tensor(note, dtype=torch.int),
         torch.tensor(start, dtype=torch.int),
         torch.tensor(end, dtype=torch.long)
         ]
        )
    return out

# Train and test splits
#data = [Mencode(x) for x in sequence]
data = torch.cat([Mencode(x) for x in sequence],dim=1)

n = int(0.9*len(data[0])) # first 90% will be train, rest val
train_data = data[:,:n]
val_data = data[:,n:]



# data loading
def get_batch(split):   
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    
    ix = torch.randint(len(data[0]) - block_size, (batch_size,))

    x = torch.stack([data[:,i:i+block_size] for i in ix])
    y = torch.stack([data[:,i+1:i+block_size+1] for i in ix])
    
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():    #TODO: fix 
    out_note = {}
    out_start = {}
    out_end = {}
    model.eval()
    for split in ['train', 'val']:
        losses_note = torch.zeros(eval_iters)
        losses_start = torch.zeros(eval_iters)
        losses_end = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            #logits, loss = model(X, Y)
            [[logits_note, loss_note],[logits_start, loss_start],[logits_end, loss_end]] = model(X, Y)
            
            losses_note[k] = loss_note.item()
            losses_start[k] = loss_start.item()
            losses_end[k] = loss_end.item()
            
        out_note[split] = losses_note.mean()
        out_start[split] = losses_start.mean()
        out_end[split] = losses_end.mean()
        
    model.train()
    return [out_note,out_start,out_end]


class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out
    
class unmaskedCrossHead(nn.Module):
    """ one head of unmasked cross-attention
        MODIFIED
    """

    def __init__(self, head_size,Head):
        super().__init__()
        #self.key = nn.Linear(n_embd, head_size, bias=False)
        self.key = Head.key
        
        self.query = nn.Linear(n_embd, head_size, bias=False)
        
        #self.value = nn.Linear(n_embd, head_size, bias=False)
        self.value = Head.value
        
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        #On ne commente pas la ligne du a l'interaction expliqué dans le rapport donc cette head est masqué

        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out
    
class MultiHeadAttentionUnmaskedCross(nn.Module):
    """ multiple heads of unmasked cross-attention in parallel 
        MODIFIED
    """

    def __init__(self, num_heads, head_size,crossingHead):
        super().__init__()
        self.heads = nn.ModuleList([unmaskedCrossHead(head_size,crossingHead.heads[_]) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x
    
class Block_self(nn.Module):
    """ Transformer block: communication followed by computation 
        MODIFIED
    """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        
        head_size = n_embd // n_head
        
        self.sa_ = MultiHeadAttention(n_head, head_size)
        self.ffwd_ = FeedFoward(n_embd)
        self.ln1_ = nn.LayerNorm(n_embd)
        self.ln2_ = nn.LayerNorm(n_embd)
        

    def forward(self, x):
        x = x + self.sa_(self.ln1_(x))
        x = x + self.ffwd_(self.ln2_(x))
        
        return x
        
class Block_cross(nn.Module): # start and end is the same block
    """
        MODIFIED
    """

    def __init__(self, n_embd, n_head, crossingHead):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        
        head_size = n_embd // n_head
        
        self.sa_ = MultiHeadAttentionUnmaskedCross(n_head, head_size, crossingHead)

        self.ffwd_ = FeedFoward(n_embd)
        self.ln1_ = nn.LayerNorm(n_embd)
        self.ln2_ = nn.LayerNorm(n_embd)
    
    def forward(self, y):

        y = y + self.sa_(self.ln1_(y))
        y = y + self.ffwd_(self.ln2_(y))

        return y

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table_note = nn.Embedding(note_vocab_size, n_embd)
        self.token_embedding_table_start = nn.Embedding(start_vocab_size, n_embd)
        self.token_embedding_table_end = nn.Embedding(end_vocab_size, n_embd)
        
        self.position_embedding_table_note = nn.Embedding(block_size, n_embd)
        self.position_embedding_table_start = nn.Embedding(block_size, n_embd)
        self.position_embedding_table_end = nn.Embedding(block_size, n_embd)
        
        LIST_BLOCK_NOTE_self  =   [Block_self(n_embd, n_head=n_head) for _ in range(n_layer)]
        LIST_BLOCK_START_self =   [Block_self(n_embd, n_head=n_head) for _ in range(n_layer)]
        LIST_BLOCK_END_self   =   [Block_self(n_embd, n_head=n_head) for _ in range(n_layer)]

        LIST_BLOCK_NOTE_cross2  =   [Block_cross(n_embd, n_head=n_head, crossingHead=LIST_BLOCK_START_self[_].sa_) for _ in range(n_layer)]
        LIST_BLOCK_START_cross2 =   [Block_cross(n_embd, n_head=n_head, crossingHead=LIST_BLOCK_END_self[_].sa_) for _ in range(n_layer)]
        LIST_BLOCK_END_cross2   =   [Block_cross(n_embd, n_head=n_head, crossingHead=LIST_BLOCK_NOTE_self[_].sa_) for _ in range(n_layer)]

        LIST_BLOCK_NOTE_cross1  =   [Block_cross(n_embd, n_head=n_head, crossingHead=LIST_BLOCK_END_cross2[_].sa_) for _ in range(n_layer)]
        LIST_BLOCK_START_cross1 =   [Block_cross(n_embd, n_head=n_head, crossingHead=LIST_BLOCK_NOTE_cross2[_].sa_) for _ in range(n_layer)]
        LIST_BLOCK_END_cross1   =   [Block_cross(n_embd, n_head=n_head, crossingHead=LIST_BLOCK_START_cross2[_].sa_) for _ in range(n_layer)]

        LIST_BLOCK_NOTE=[]
        for x in range(n_layer):
            LIST_BLOCK_NOTE.append(LIST_BLOCK_NOTE_self[x])
            LIST_BLOCK_NOTE.append(LIST_BLOCK_NOTE_cross1[x])
            LIST_BLOCK_NOTE.append(LIST_BLOCK_NOTE_cross2[x])

        LIST_BLOCK_START=[]
        for x in range(n_layer):
            LIST_BLOCK_START.append(LIST_BLOCK_START_self[x])
            LIST_BLOCK_START.append(LIST_BLOCK_START_cross1[x])
            LIST_BLOCK_START.append(LIST_BLOCK_START_cross2[x])

        LIST_BLOCK_END=[]
        for x in range(n_layer):
            LIST_BLOCK_END.append(LIST_BLOCK_END_self[x])
            LIST_BLOCK_END.append(LIST_BLOCK_END_cross1[x])
            LIST_BLOCK_END.append(LIST_BLOCK_END_cross2[x])
        
        self.blocks_note = nn.Sequential(*LIST_BLOCK_NOTE)
        self.blocks_start = nn.Sequential(*LIST_BLOCK_START) 
        self.blocks_end = nn.Sequential(*LIST_BLOCK_END)
        
        self.ln_f_note = nn.LayerNorm(n_embd) # final layer norm
        self.ln_f_start = nn.LayerNorm(n_embd) # final layer norm
        self.ln_f_end = nn.LayerNorm(n_embd) # final layer norm
        
        self.lm_head_note = nn.Linear(n_embd, note_vocab_size)
        self.lm_head_start = nn.Linear(n_embd, start_vocab_size)
        self.lm_head_end = nn.Linear(n_embd, end_vocab_size)

        # better init, not covered in the original GPT video, but important, will cover in followup video
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):

        note,start,end=torch.unbind(idx, dim=1)
        # séparer les donnés note,start,end en 3 tenseur (B,T)
        
        B_note,T_note = note.shape
        B_start,T_start = start.shape
        B_end,T_end = end.shape
        
        tok_emb_note = self.token_embedding_table_note(note) # (B,T,C)
        pos_emb_note = self.position_embedding_table_note(torch.arange(T_note, device=device)) # (T,C)
        
        tok_emb_start = self.token_embedding_table_start(start) # (B,T,C)
        pos_emb_start = self.position_embedding_table_start(torch.arange(T_start, device=device)) # (T,C)
        
        tok_emb_end = self.token_embedding_table_end(end) # (B,T,C)
        pos_emb_end = self.position_embedding_table_end(torch.arange(T_end, device=device)) # (T,C)
        
        x = tok_emb_note + pos_emb_note # (B,T,C)
        y = tok_emb_start + pos_emb_start # (B,T,C)
        z = tok_emb_end + pos_emb_end # (B,T,C)
        
        x = self.blocks_note(x) # (B,T,C)  # 
        y = self.blocks_start(y) # (B,T,C)  #
        z = self.blocks_end(z) # (B,T,C)  #
        
        x = self.ln_f_note(x) # (B,T,C)
        y = self.ln_f_start(y) # (B,T,C)
        z = self.ln_f_end(z) # (B,T,C)
        
        logits_note = self.lm_head_note(x) # (B,T,vocab_size)
        logits_start = self.lm_head_start(y) # (B,T,vocab_size)
        logits_end = self.lm_head_end(z) # (B,T,vocab_size)
        
        
        
        if targets is None:
            loss_note = None
            loss_start = None
            loss_end = None
        else:

            targets_note,targets_start,targets_end=torch.unbind(targets, dim=1)
            
            targets_note = targets_note.to(torch.int64).contiguous()
            targets_start = targets_start.to(torch.int64).contiguous()
            targets_end = targets_end.to(torch.int64).contiguous()
            
            B_note, T_note, C_note = logits_note.shape
            logits_note = logits_note.view(B_note*T_note, C_note)
            targets_note = targets_note.view(B_note*T_note)
            loss_note = F.cross_entropy(logits_note, targets_note)
            
            B_start, T_start, C_start = logits_start.shape
            logits_start = logits_start.view(B_start*T_start, C_start)
            targets_start = targets_start.view(B_start*T_start)
            loss_start = F.cross_entropy(logits_start, targets_start)
            
            B_end, T_end, C_end = logits_end.shape
            logits_end = logits_end.view(B_end*T_end, C_end)
            targets_end = targets_end.view(B_end*T_end)
            loss_end = F.cross_entropy(logits_end, targets_end)

        return [[logits_note, loss_note],[logits_start, loss_start],[logits_end, loss_end]]

    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            
            idx_cond = idx[:,:,-block_size:]
            #de même on sépare les données en 3 tenseur
            
            # get the predictions
            [[logits_note, loss_note],[logits_start, loss_start],[logits_end, loss_end]] = self(idx_cond)
            
            # focus only on the last time step
            logits_note = logits_note[:, -1, :] # becomes (B, C)
            logits_start = logits_start[:, -1, :] # becomes (B, C)
            logits_end = logits_end[:, -1, :] # becomes (B, C)
            
            # apply softmax to get probabilities
            probs_note = F.softmax(logits_note, dim=-1) # (B, C)
            probs_start = F.softmax(logits_start, dim=-1) # (B, C)
            probs_end = F.softmax(logits_end, dim=-1) # (B, C)
            
            # sample from the distribution
            idx_next_note = torch.multinomial(probs_note, num_samples=1) # (B, 1)
            idx_next_start = torch.multinomial(probs_start, num_samples=1) # (B, 1)
            idx_next_end = torch.multinomial(probs_end, num_samples=1) # (B, 1)

            nextToken=torch.stack([idx_next_note,idx_next_start,idx_next_end])
            nextToken=torch.reshape(nextToken,(1,3,1)) 

            # append sampled index to the running sequence
            idx = torch.cat((idx, nextToken), dim=2) # (B, T+1)
        return idx

model = GPTLanguageModel()

m = model.to(device)

# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

history=[]

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses_note,losses_start,losses_end = estimate_loss()
        print(f"""step {iter}:------------------------------------------------------------
                  \n train_note loss\t{losses_note['train']:.4f}, train_start loss\t{losses_start['train']:.4f},train_end loss\t{losses_end['train']:.4f}
                  \n val_note loss\t{losses_note['val']:.4f},val_start loss\t{losses_start['val']:.4f},val_end loss\t{losses_end['val']:.4f}
              """)

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    #logits, loss = model(xb, yb)
    
    [[logits_note, loss_note],[logits_start, loss_start],[logits_end, loss_end]] = model(xb, yb)
    
    history.append([loss_note,loss_start,loss_end])
    
    optimizer.zero_grad(set_to_none=True)
    loss_note.backward()
    loss_start.backward()
    loss_end.backward()
    
    optimizer.step()
    
# generate from the model
context = torch.zeros((3, 1), dtype=torch.int, device=device)
context[0]=random.randint(20, 100) # start with a random note 
context[2]=random.randint(1, 99) #should also end note at 0.0
context=torch.stack([context])

print("generating")

note_out, start_out, end_out = m.generate(context, max_new_tokens=100)[0]

Note_decode = lambda l: [note_dict_decode[i.tolist()] for i in l]
Start_decode = lambda l: [start_dict_decode[i.tolist()] for i in l]
End_decode = lambda l: [end_dict_decode[i.tolist()] for i in l] 

note_=Note_decode(note_out)
start_=Start_decode(start_out)
end_=End_decode(end_out)

length=len(note_)
music=[[note_[x],start_[x],end_[x]] for x in range(length)]
OutMidi(music)

In [146]:
torch.save(model.state_dict(), "modelWeight")

In [ ]:
#model = TheModelClass(*args, **kwargs)
#model.load_state_dict(torch.load(PATH, weights_only=True))
#model.eval()

In [ ]:
import matplotlib.pyplot as plt

import numpy as np
history=torch.tensor(history).tolist()
xpoints = [x for x in range(len(history))]
ypoints = [x[0] for x in history]

plt.plot(xpoints, ypoints)
plt.show()

ypoints = [x[1] for x in history]

plt.plot(xpoints, ypoints)
plt.show()

ypoints = [x[2] for x in history]

plt.plot(xpoints, ypoints)
plt.show()
